In [50]:
import numpy as np
import pandas as pd
import os
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


__mapping for categorical data conversion from orig mental health dataset__
mapGender = {'Male':0, 'Female':1, 'Other':2}  
mapMarital = {'Single':0, 'Divorced':1, 'Married':2}  
mapEdu = {'High School':0, 'Bachelor':1, 'Master':2, 'PhD':3}  
mapEmpl = {'Student':0, 'Unemployed':1, 'Employed':2, 'Self-Employed':3}  
maps = [mapGender, mapMarital, mapEdu, mapEmpl]  

In [59]:
mapGender = {'Male':0, 'Female':1, 'Other':2}
mapGenre = {"Mobile Games":0, "MOBA":1, "FPS":2, "Battle Royale":3, "MMO":4, "RPG":5, "Strategy":6}
mapPlatform = {"PC":0, "Multi-platform":1, "Console":2, "Mobile":3}
mapSQual = {"Insomnia":0, "Very Poor":1, "Poor":2, "Fair":3, "Good":4}
mapFreq = {"Never":0, "Rarely":1, "Sometimes":2, "Often":3, "Always":4}
mapAperf = {"Failing":0, "Poor":1, "Below Average":2, "Average":3, "Good":4, "Excellent":5}
mapMood = {"Depressed":0, "Anxious":1, "Angry":2, "Irritable":3, "Withdrawn":4, "Restless":5, "Normal":6, "Excited":7, "Euphoric":8}
mapAddict = {"Low":0, "Moderate":1, "High":2, "Severe":3}
mapBinary = {"TRUE":1, "FALSE":0}
maps = [mapGender, mapGenre, mapPlatform, mapSQual, mapFreq, mapAperf, mapMood, mapFreq, mapAddict]

In [72]:
def readData(datafile):
    # get the current directory: directory name for the abs path of the curr file
    dirpath = os.getcwd()
    abspath = dirpath + "\\" + datafile

    # read data into a pandas dataframe. data file is comma separated so use read_csv
    df = pd.read_csv(abspath)
     
    return df


# convert categorical data to numerical 
def convertData(data, drop:list[str], maps:list[dict], export:tuple[bool, str]):
    # drop indicated columns (should be a list of header names)
    if drop:
        for i in range(len(drop)):
            data = data.drop(columns=drop[i])

    if maps:
        idx=0
        for feat in data.columns:
            if data[feat].dtype == 'string':
                #print(f"Converting column {feat} with map {idx}")
                data[feat] = data[feat].map(maps[idx])
                idx += 1
            elif data[feat].dtype == 'bool':
                data[feat] = data[feat].astype(int)
                        
        if (export[0]):
            data.to_csv(export[1])

    return data

In [75]:
filename = "GamingandMentalHealth.csv"  
data = readData(filename)

# drop primary game for now 
dropCols = ["record_id", "primary_game"]
exportTo = (True, "GamingandMentalHealth_conv_noidprimgame.csv")
data_conv = convertData(data.copy(), dropCols, maps, exportTo)

#print(data_conv.head(1))

### Need to fill in missing data values and convert the primary game feature before doing spearman or tSNE or anything else

In [78]:
spearman = data_conv.corr(method='spearman')
print(spearman)
spearman.to_csv("spearman_corr_gaming.csv")

                                       age    gender  daily_gaming_hours  \
age                               1.000000  0.018518           -0.013442   
gender                            0.018518  1.000000           -0.028432   
daily_gaming_hours               -0.013442 -0.028432            1.000000   
game_genre                       -0.012297 -0.011469            0.028563   
gaming_platform                   0.022091  0.024227           -0.019794   
sleep_hours                       0.020552  0.025449           -0.777688   
sleep_quality                     0.052844  0.040515           -0.532374   
sleep_disruption_frequency       -0.032659 -0.062881           -0.028450   
academic_work_performance        -0.020869  0.030830           -0.551445   
grades_gpa                       -0.023733  0.021369            0.025749   
work_productivity_score          -0.033703  0.000349           -0.002656   
mood_state                        0.008468 -0.026032           -0.315635   
mood_swing_f

In [79]:
data_conv = data_conv.to_numpy()
print(data_conv)

x = data_conv[:, 0:-1]       # data is all features
target = data_conv[:,-1]     # target is mental_health_risk

scaler = StandardScaler()
xscaled = scaler.fit_transform(x)

ndims = 2 # dims of the embedded space
tsne = TSNE(ndims)
result = tsne.fit_transform(xscaled)
result_df = pd.DataFrame({'TSNE_col1': result[:,0], 'TSNE_col2': result[:,1], 'GamingMentalHealth' : target})
fig, ax = plt.subplots()
ax.scatter(x=result_df['TSNE_col1'], y=result_df['TSNE_col2'], c=result_df['GamingMentalHealth'])
plt.title("Gaming Mental health risk data in 2D")
plt.show()




[[ 17.     0.    11.1  ... 383.7    3.     3.  ]
 [ 21.     0.     3.   ...  46.64   1.     0.  ]
 [ 23.     0.     7.6  ... 100.81   6.     3.  ]
 ...
 [ 23.     0.     7.3  ...  88.6    5.     2.  ]
 [ 18.     0.     3.1  ...  22.02   8.     0.  ]
 [ 29.     0.     3.5  ...  25.2   19.     0.  ]]


ValueError: Input X contains NaN.
TSNE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values